# 2. Post-processing profiling data

This notebook is a tour of the analysis API: `ProfilingH5Reader` and the
`MPIRegion` / `Region` objects it hands out.

The one thing worth memorising: **every duration and timestamp in this API is in
seconds**, converted from the nanoseconds stored on disk.

In [ ]:
import tempfile
import time
from pathlib import Path

from scope_profiler import ProfileManager, ProfilingH5Reader

WORKDIR = Path(tempfile.mkdtemp(prefix="scope-profiler-tutorial-"))
DATA_FILE = WORKDIR / "profiling_data.h5"

ProfileManager.setup(file_path=str(DATA_FILE))


@ProfileManager.profile("assemble")
def assemble(scale):
    time.sleep(0.004 * scale)


with ProfileManager.profile_region("setup"):
    time.sleep(0.02)

for step in range(5):
    with ProfileManager.profile_region("timestep"):
        assemble(scale=1 + step % 3)
        with ProfileManager.profile_region("solve"):
            time.sleep(0.003)

ProfileManager.finalize(verbose=False)
reader = ProfilingH5Reader(DATA_FILE)
reader

## The reader is a mapping of regions

A reader behaves like an ordered mapping from region name to region, so the
usual Python idioms work.

In [ ]:
print("regions:", reader.region_names)
print("count:", len(reader))
print("'solve' recorded?", "solve" in reader)
print("ranks in file:", reader.num_ranks)

for region in reader:
    print(f"  {region.name:<10} {region.num_calls:>3} calls")

Asking for something that is not there tells you what *is* there:

In [ ]:
try:
    reader["sovle"]  # typo
except KeyError as exc:
    print(exc)

## Two levels: MPIRegion and Region

`reader["solve"]` returns an **`MPIRegion`** — the region across every rank that
recorded it. Indexing it by rank gives a **`Region`**, the timings from that one
rank.

For a serial run there is only rank 0, but the two levels are the same API you
use for an MPI run.

In [ ]:
timestep = reader["timestep"]

print("aggregated over ranks")
print("  ranks       :", timestep.ranks)
print("  num_calls   :", timestep.num_calls)
print("  total   [s] :", timestep.total_duration)
print("  average [s] :", timestep.average_duration)
print("  min/max [s] :", timestep.min_duration, timestep.max_duration)
print("  std     [s] :", timestep.std_duration)

rank0 = timestep[0]
print("\nrank 0 only")
print("  durations [s]:", rank0.durations)
print("  start times [s]:", rank0.start_times)
print("  summary:", rank0.get_summary())

Per-rank breakdowns are available as dictionaries keyed by rank, which is where
load imbalance shows up in an MPI run:

In [ ]:
print("calls per rank :", timestep.num_calls_per_rank())
print("total per rank :", timestep.total_durations())
print("avg per rank   :", timestep.average_durations())
print("max per rank   :", timestep.max_durations())

## Summaries and DataFrames

`summary()` returns one dict per region, aggregated over ranks.
`print_summary()` formats the same data as a table.

In [ ]:
reader.print_summary()

In [ ]:
for row in reader.summary():
    print(row)

With pandas installed (it comes with the `pproc` extra), `to_dataframe()` gives
you the same data ready for sorting, filtering and plotting.

In [ ]:
frame = reader.to_dataframe()
frame.sort_values("total_duration", ascending=False)

`per_rank=True` emits one row per (region, rank) instead — the shape you want
for load-balance analysis.

In [ ]:
reader.to_dataframe(per_rank=True)

In [ ]:
# Share of total recorded time per region.
frame = frame.assign(share=lambda df: df.total_duration / df.total_duration.sum())
frame[["name", "num_calls", "total_duration", "share"]].sort_values(
    "share", ascending=False
)

## Selecting regions

`get_regions()`, `summary()`, `to_dataframe()` and the plotting functions all
take `include` / `exclude`, which are **regular expressions** matched against
the region name (via `re.match`, so they anchor at the start).

In [ ]:
print([region.name for region in reader.get_regions(include="s")])
print([region.name for region in reader.get_regions(exclude=["setup", "assemble"])])
print([row["name"] for row in reader.summary(include=["solve", "timestep"])])

## Run metadata

Each file carries the environment it was recorded in: the host and CPU, the
loaded environment modules, the Slurm job it ran under, and selected
environment variables such as `PATH` and `VIRTUAL_ENV`. This is what tells two
otherwise identical runs apart months later.

In [ ]:
for key, value in sorted(reader.metadata.items()):
    # PATH and friends run to thousands of characters; clip them for display.
    text = str(value)
    if len(text) > 70:
        text = text[:70] + " […]"
    print(f"{key:>24}: {text}")

`scope-profiler inspect` prints the same metadata grouped and readable —
together with per-region statistics — straight from the command line:

```bash
scope-profiler inspect profiling_data.h5
scope-profiler inspect profiling_data.h5 --metadata-only
```

`write_metadata_json()` exports it instead, with one entry per file and no
clipping of long values:

In [ ]:
from scope_profiler.inspection import write_metadata_json

payload = write_metadata_json(DATA_FILE, WORKDIR / "metadata.json")
print(sorted(payload["files"][0]["metadata"])[:8])

## Exporting statistics

`collect_region_statistics()` builds a JSON-ready dict (durations in seconds,
with a `units` field spelling that out), and `write_region_statistics_json()`
writes it to a file. Both accept several readers at once for run comparisons.

In [ ]:
from scope_profiler import collect_region_statistics, write_region_statistics_json

stats = collect_region_statistics(reader, include="solve")
print(stats["units"])
print(stats["files"][0]["region_statistics"]["solve"])

stats_path = WORKDIR / "region_statistics.json"
write_region_statistics_json(reader, stats_path)
print("wrote", stats_path)

## Regions without timing

When a run is profiled with `time_trace=False` the profiler records only how
often each region was entered. Those regions still appear, with `has_timing`
`False` and zero durations, so summaries and plots stay well-defined.

In [ ]:
COUNTS_FILE = WORKDIR / "counts_only.h5"
ProfileManager.setup(time_trace=False, file_path=str(COUNTS_FILE))

for _ in range(4):
    with ProfileManager.profile_region("cheap_call"):
        pass

ProfileManager.finalize(verbose=False)

counts = ProfilingH5Reader(COUNTS_FILE)
region = counts["cheap_call"]
print("num_calls  :", region.num_calls)
print("has_timing :", region.has_timing)
print("durations  :", region.durations)

## Next

[3. Visualizing results](03_visualization.ipynb) turns these numbers into Gantt,
flame and duration charts, and
[5. Custom analysis](05_custom_analysis.ipynb) goes below the summaries to the
individual calls.